# UTICA Checkpoint

UTICA is an additional pre-trained checkpoint for the `MantisV1` architecture, obtained with a self-distillation pre-training recipe.

Unlike our other checkpoints, it is not hosted in the `paris-noah` collection but in the [`fegounna/Utica`](https://huggingface.co/fegounna/Utica) repository, so we pin an explicit revision to keep the results reproducible.

## Read data

demonstration data set from the UCR collection

In [1]:
import numpy as np

data = [np.load(f'../data/GestureMidAirD1/{variable}_{set_name}.npy')
        for variable in ['X', 'y'] for set_name in ['train', 'test']]

X_train, X_test, y_train, y_test = data

print("X_train dims: ", X_train.shape)
print("X_test dims: ", X_test.shape)

X_train dims:  (208, 1, 360)
X_test dims:  (130, 1, 360)


in order to apply the foundation model to your data, `X_train` and `X_test` should be of the shape `(n_samples, n_channels=1, seq_len)`, where `seq_len` is a multiple of 32

if original sequence length is different, resize it, for example, using the following function:

In [2]:
import torch
import torch.nn.functional as F

def resize(X):
    X_scaled = F.interpolate(torch.tensor(X, dtype=torch.float), size=512, mode='linear', align_corners=False)
    return X_scaled.numpy()
    
X_train, X_test = resize(X_train), resize(X_test)

print("X_train dims: ", X_train.shape)
print("X_test dims: ", X_test.shape)

X_train dims:  (208, 1, 512)
X_test dims:  (130, 1, 512)


## Load the checkpoint

The checkpoint is loaded exactly like the others, except that we pass the `revision` of the `fegounna/Utica` repository:

In [3]:
from mantis.architecture import MantisV1

UTICA_REPO = 'fegounna/Utica'
UTICA_REVISION = '3cff4f954191b5bf9839b7a41117e2b24e7693ab'

device = 'cpu' # set device

network = MantisV1(device=device)
network = network.from_pretrained(UTICA_REPO, revision=UTICA_REVISION)

print("embedding dimension: ", network.hidden_dim)

embedding dimension:  256


Two remarks on this checkpoint.

First, the repository stores the weights as `pytorch_model.bin` and ships no `config.json`, so the architecture arguments are not read from the Hub but taken from the `MantisV1` object that you construct. All the defaults of `MantisV1` match the ones this checkpoint was pre-trained with, so there is nothing to pass besides `device`.

Second, the weights are loaded non-strictly, and the mismatch is expected in both directions: the pre-training projector `prj` is absent from the checkpoint because it is not needed to extract features, while the checkpoint still carries the iBOT mask token and the final layer normalization of the UTICA objective, neither of which belongs to the encoder. Ignoring them is exactly what we want, and the resulting embeddings are unaffected.

## Evaluation protocol

In [4]:
from mantis.trainer import MantisTrainer
from sklearn.ensemble import RandomForestClassifier

def evaluate_model(network, layer_idx):
    model = MantisTrainer(device=device, network=network) # init trainer
    Z_train = model.transform(X_train)
    Z_test = model.transform(X_test)

    predictor = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
    predictor.fit(Z_train, y_train)

    y_pred = predictor.predict(Z_test)
    print(f'Accuracy at the layer {layer_idx}: {np.mean(y_test == y_pred)}')

## Results

As for the other checkpoints, it is worth trying each transformer layer's output as the final embedding, as well as both output tokens. By default, the classification token (`output_token='cls_token'`) is used.

In [5]:
for layer_idx in range(6):
    network = MantisV1(device=device, return_transf_layer=layer_idx) # init model
    network = network.from_pretrained(UTICA_REPO, revision=UTICA_REVISION) # load weights
    evaluate_model(network, layer_idx)

Accuracy at the layer 0: 0.676923076923077


Accuracy at the layer 1: 0.6846153846153846


Accuracy at the layer 2: 0.6846153846153846


Accuracy at the layer 3: 0.6923076923076923


Accuracy at the layer 4: 0.6923076923076923


Accuracy at the layer 5: 0.6846153846153846


Now the same sweep, using the concatenation of the classification token and the mean over the non-classification tokens (`output_token='combined'`), which doubles the embedding dimension:

In [6]:
for layer_idx in range(6):
    network = MantisV1(device=device, return_transf_layer=layer_idx, output_token='combined') # init model
    network = network.from_pretrained(UTICA_REPO, revision=UTICA_REVISION) # load weights
    evaluate_model(network, layer_idx)

Accuracy at the layer 0: 0.676923076923077


Accuracy at the layer 1: 0.6538461538461539


Accuracy at the layer 2: 0.7230769230769231


Accuracy at the layer 3: 0.7


Accuracy at the layer 4: 0.7


Accuracy at the layer 5: 0.6846153846153846


On this data set, the best combination is the third transformer layer (`return_transf_layer=2`) together with `output_token='combined'`, which is also what we observed on larger benchmarks. As always, both arguments are worth tuning on your own data.